In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from google import genai

C:\Users\Dell\AppData\Local\Temp\ipykernel_9384\1230240314.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


**Sample Knowledge Base**

In [3]:
text = """
Artificial Intelligence is the simulation of human intelligence by machines.

Machine Learning is a subset of AI that enables computers to learn from data.

Deep Learning is a subset of Machine Learning based on neural networks.

Natural Language Processing enables computers to understand and generate human language.

FAISS is a vector database used for similarity search.

RAG stands for Retrieval-Augmented Generation.

Gemini is a Large Language Model developed by Google.
"""

**Split the Text**

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = splitter.split_text(text)

print(chunks)

['Artificial Intelligence is the simulation of human intelligence by machines.', 'Machine Learning is a subset of AI that enables computers to learn from data.', 'Deep Learning is a subset of Machine Learning based on neural networks.', 'Natural Language Processing enables computers to understand and generate human language.', 'FAISS is a vector database used for similarity search.', 'RAG stands for Retrieval-Augmented Generation.', 'Gemini is a Large Language Model developed by Google.']


**Create Embeddings**

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Build the Vector Database**

In [6]:
vector_db = FAISS.from_texts(
    chunks,
    embedding=embeddings
)

print("Vector database created successfully!")

Vector database created successfully!


**part 2 Build the Complete RAG Pipeline**

Knowledge Base  
   ↓  
Chunks   
   ↓   
Embeddings  
   ↓  
FAISS   
   ↓  
Retriever   
   ↓    
Gemini  
   ↓  
Final Answer

**Create a Retriever**

In [7]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 2}
)

print("Retriever created successfully!")

Retriever created successfully!


**query**

In [8]:
query = "What is RAG?"

**Retrieve Relevant Chunks**

In [10]:
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}")
    print(doc.page_content)


Document 1
RAG stands for Retrieval-Augmented Generation.

Document 2
Gemini is a Large Language Model developed by Google.


**Combine Retrieved Context**

In [11]:
context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

print(context)

RAG stands for Retrieval-Augmented Generation.

Gemini is a Large Language Model developed by Google.


**Create the Prompt**

In [16]:
prompt = f"""
You are an AI assistant.

Answer the question ONLY using the information provided below.

If the answer is not found, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""

**Send to Gemini**

In [1]:
client = genai.Client(api_key="from google studio, your API key")
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

NameError: name 'genai' is not defined

**without RAG**

User   
  ↓  
Gemini  
  ↓  
Answer (based on its own knowledge)

**with RAG**

User  
  ↓   
Retriever   
  ↓   
Relevant Context  
  ↓  
Gemini  
  ↓   
Answer based on Context